# Deep Learning Lab: all experiments in Keras
Every experiment is one self-contained cell: run any of them on its own.

**Runtime:** Colab, TensorFlow/Keras 3. Use a **GPU** runtime for Ex-6, 7, 12(b) and 14.

**Data you must provide yourself (Kaggle):** Ex-6 vehicle images (`DATASET_PATH`), Ex-7 retinopathy images + `trainLabels.csv` (`IMG_DIR`, `LABELS_CSV`). Everything else downloads automatically; Ex-9, 10 and 14 ask you to upload a CSV if the download is blocked.

In [ ]:
%pip -q install tensorflow-datasets yfinance

## Ex-1  Understanding the Perceptron
**Aim:** Understand the perceptron by classifying Iris Setosa vs Versicolor with a single sigmoid neuron in Keras.

In [ ]:
# Ex-1  Understanding the Perceptron (Iris: Setosa vs Versicolor)
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 2017
keras.utils.set_random_seed(SEED)

# 1. Load data: keep the two linearly separable classes (0 = Setosa, 1 = Versicolor)
iris = load_iris()
idx = np.where(iris.target < 2)
X = iris.data[idx].astype("float32")
y = iris.target[idx].astype("float32")

# 2. Plot two of the four variables
plt.scatter(X[y == 0][:, 0], X[y == 0][:, 2], color="green", label="Iris-Setosa")
plt.scatter(X[y == 1][:, 0], X[y == 1][:, 2], color="red", label="Iris-Versicolour")
plt.xlabel("sepal length (cm)"); plt.ylabel("petal length (cm)"); plt.legend(); plt.show()

# 3. Train / validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED)
scaler = StandardScaler().fit(X_train)                 # scaled inputs keep the sigmoid out of saturation
X_train, X_val = scaler.transform(X_train), scaler.transform(X_val)

# 4. Perceptron = one neuron: weighted sum + bias -> sigmoid (output > 0.5 means class 1)
model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.1),
              loss="binary_crossentropy",
              metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])
model.summary()

# 5. Train
history = model.fit(X_train, y_train, epochs=15, batch_size=8,
                    validation_data=(X_val, y_val), verbose=2)

# 6. Learned parameters and final score
w, b = model.layers[0].get_weights()
print("Weights:", w.ravel(), "| bias:", b)
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f"Validation loss: {val_loss:.4f} | Validation accuracy: {val_acc:.4f}")

# 7. Plot training loss and validation accuracy
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].plot(history.history["loss"]); ax[0].set_title("Training loss"); ax[0].set_xlabel("epochs")
ax[1].plot(history.history["val_accuracy"]); ax[1].set_title("Validation accuracy"); ax[1].set_xlabel("epochs")
plt.tight_layout(); plt.show()


## Ex-2  Perceptron on the Diabetes dataset
**Aim:** Build a perceptron in Keras for the Diabetes dataset and report accuracy, precision, recall and F1.

In [ ]:
# Ex-2  Perceptron on the Diabetes dataset (binary: progression > 140 or not)
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

keras.utils.set_random_seed(42)

# 1. Load data
diabetes = datasets.load_diabetes()
X, y = diabetes.data, diabetes.target

# 2. Preprocess: use one feature (BMI, column 2) and binarise the target
X = X[:, np.newaxis, 2]
y = (y > 140).astype("float32")

# 3. Split and standardise (fit scaler on train only)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)

# 4. Perceptron: a single sigmoid neuron
model = keras.Sequential([
    keras.Input(shape=(1,)),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.05),
              loss="binary_crossentropy",
              metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])

history = model.fit(X_train, y_train, epochs=100, batch_size=32,
                    validation_split=0.2, verbose=0)

# 5. Evaluate on the test set
y_pred = (model.predict(X_test, verbose=0).ravel() > 0.5).astype(int)
print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1 score :", round(f1_score(y_test, y_pred, zero_division=0), 4))

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].plot(history.history["loss"], label="train"); ax[0].plot(history.history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epochs"); ax[0].legend()
ax[1].plot(history.history["accuracy"], label="train"); ax[1].plot(history.history["val_accuracy"], label="val")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epochs"); ax[1].legend()
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred)).plot(ax=ax[2], colorbar=False)
ax[2].set_title("Confusion matrix (test)")
plt.tight_layout(); plt.show()


## Ex-3  Single-layer neural network
**Aim:** Build and train a neural network with one hidden layer on the two-circles dataset.

In [ ]:
# Ex-3  Single-hidden-layer neural network on the "two circles" dataset
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

SEED = 2017
keras.utils.set_random_seed(SEED)

# 1. Data
X, y = make_circles(n_samples=400, factor=0.3, noise=0.05, random_state=SEED)
outer, inner = y == 0, y == 1
plt.title("Two Circles")
plt.plot(X[outer, 0], X[outer, 1], "ro"); plt.plot(X[inner, 0], X[inner, 1], "bo"); plt.show()

X = (X + 1).astype("float32")
y = y.astype("float32")
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED)

# 2. Model: 2 inputs -> 50 sigmoid hidden units -> 1 sigmoid output
n_hidden, n_epochs, learning_rate = 50, 1000, 1.0
model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(n_hidden, activation="sigmoid", use_bias=False,
                 kernel_initializer=keras.initializers.RandomNormal(0.0, 1.0)),   # weights ~ N(0, 1), as in the manual
    layers.Dense(1, activation="sigmoid", use_bias=False,
                 kernel_initializer=keras.initializers.RandomNormal(0.0, 1.0)),
])
model.compile(optimizer=keras.optimizers.SGD(learning_rate=learning_rate),
              loss="mse",
              metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])
model.summary()

# 3. Print validation stats every 100 epochs
class Report(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if epoch % 100 == 0:
            print(f"Epoch: {epoch:>4} ; Validation loss: {logs['val_loss']:.4f} ; "
                  f"Validation accuracy: {logs['val_accuracy']:.4f}")

# Full-batch gradient descent (batch = whole training set), as in the manual
history = model.fit(X_train, y_train, epochs=n_epochs, batch_size=len(X_train),
                    validation_data=(X_val, y_val), verbose=0, callbacks=[Report()])

loss, acc = model.evaluate(X_val, y_val, verbose=0)
print(f"\nFinal validation loss: {loss:.4f} | accuracy: {acc:.4f}")

# 4. Curves and decision boundary
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(history.history["loss"], label="train"); ax[0].plot(history.history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epochs"); ax[0].legend()
ax[1].plot(history.history["accuracy"], label="train"); ax[1].plot(history.history["val_accuracy"], label="val")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epochs"); ax[1].legend()
gx, gy = np.meshgrid(np.linspace(X[:, 0].min() - .2, X[:, 0].max() + .2, 200),
                     np.linspace(X[:, 1].min() - .2, X[:, 1].max() + .2, 200))
zz = model.predict(np.c_[gx.ravel(), gy.ravel()].astype("float32"), verbose=0).reshape(gx.shape)
ax[2].contourf(gx, gy, zz > 0.5, alpha=0.3, cmap="coolwarm")
ax[2].scatter(X_val[:, 0], X_val[:, 1], c=y_val, cmap="coolwarm", edgecolor="k")
ax[2].set_title("Decision boundary (validation points)")
plt.tight_layout(); plt.show()


## Ex-4  Multi-layer neural network
**Aim:** Build a multi-layer network for Wine Quality regression with early stopping and checkpointing.

In [ ]:
# Ex-4  Multi-layer neural network (regression on Wine Quality - red)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 2017
keras.utils.set_random_seed(SEED)

def load_wine():
    urls = ["https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv",
            "https://raw.githubusercontent.com/shrikant-temburwar/Wine-Quality-Dataset/master/winequality-red.csv"]
    for u in urls:
        try:
            return pd.read_csv(u, sep=";")
        except Exception as e:
            print("could not read", u, "->", type(e).__name__)
    # Last resort: upload winequality-red.csv to Colab and use pd.read_csv("winequality-red.csv", sep=";")
    raise RuntimeError("Wine data not reachable")

data = load_wine()
y = data["quality"]
X = data.drop(["quality"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print("Average quality (train): {:.4f}".format(y_train.mean()))

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

# Baseline: always predict the mean quality
print("Baseline MSE:", np.mean((y_test - y_train.mean()) ** 2))

# Network: 200 -> 25 -> 1 (linear output for regression)
model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    layers.Dense(200, activation="relu"),
    layers.Dense(25, activation="relu"),
    layers.Dense(1, activation="linear"),
])
model.compile(optimizer=keras.optimizers.Adam(), loss="mse", metrics=["mae"])
model.summary()

# Early stopping + keep the best weights (what ModelCheckpoint did in the manual)
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=30, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint("multi_layer_best_model.keras", monitor="val_loss",
                                    save_best_only=True, verbose=0),
]
history = model.fit(X_train, y_train.values, batch_size=64, epochs=500,
                    validation_split=0.2, callbacks=callbacks, verbose=2)

# Evaluate the best model on the test set
mse, mae = model.evaluate(X_test, y_test.values, verbose=0)
pred = model.predict(X_test, verbose=0).ravel()
acc = np.mean(np.round(pred) == y_test.values)   # quality is an integer, so round the prediction
print(f"Test MSE: {mse:.4f} | Test MAE: {mae:.4f} | Rounded-quality accuracy: {acc*100:.2f}%")
print("Benchmark accuracy on this dataset: 62.4%")

plt.plot(history.history["loss"], label="train"); plt.plot(history.history["val_loss"], label="validation")
plt.title("Multi-layer network: MSE loss"); plt.xlabel("epochs"); plt.legend(); plt.show()


## Ex-5  Activation functions
**Aim:** Compare sigmoid and ReLU in a deep network on MNIST (loss per batch, weights per layer).

In [ ]:
# Ex-5  Experiment with activation functions: sigmoid vs ReLU on MNIST
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

SEED = 2022
keras.utils.set_random_seed(SEED)

# 1. Plot the activation functions themselves
z = np.linspace(-6, 6, 200)
plt.figure(figsize=(10, 3))
for i, (name, f) in enumerate({"sigmoid": lambda v: 1 / (1 + np.exp(-v)),
                               "tanh": np.tanh,
                               "ReLU": lambda v: np.maximum(0, v)}.items()):
    plt.subplot(1, 3, i + 1); plt.plot(z, f(z)); plt.title(name); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

# 2. Data: MNIST, normalised and flattened
(X_train, y_train), (X_val, y_val) = keras.datasets.mnist.load_data()
plt.figure(figsize=(12, 3))
for label in range(10):
    plt.subplot(1, 10, label + 1); plt.axis("off")
    plt.title(f"{label}: ({int((y_train == label).sum())})", fontsize=7)
    plt.imshow(X_train[np.where(y_train == label)[0][0]], cmap="gray")
plt.show()

X_train = X_train.reshape(-1, 784).astype("float32") / 255.0
X_val = X_val.reshape(-1, 784).astype("float32") / 255.0
y_train = keras.utils.to_categorical(y_train, 10)
y_val = keras.utils.to_categorical(y_val, 10)

# 3. Same deep network, two activations
def build(activation):
    m = keras.Sequential([keras.Input(shape=(784,))])
    for units in [700, 700, 700, 700, 700, 350, 100]:
        m.add(layers.Dense(units, activation=activation))
    m.add(layers.Dense(10, activation="softmax"))
    m.compile(loss="categorical_crossentropy", optimizer="sgd", metrics=["accuracy"])
    return m

class BatchLoss(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.losses = []
    def on_train_batch_end(self, batch, logs=None):
        self.losses.append(logs.get("loss"))

models, losses = {}, {}
for act in ["sigmoid", "relu"]:
    m, cb = build(act), BatchLoss()
    m.fit(X_train, y_train, epochs=10, batch_size=256, validation_split=0.2,
          callbacks=[cb], verbose=2)
    models[act], losses[act] = m, cb.losses
    print(act, "validation-set accuracy:", round(m.evaluate(X_val, y_val, verbose=0)[1], 4))

# 4. Loss per batch
for act in losses:
    plt.plot(losses[act], label=act)
plt.title("Losses for sigmoid and ReLU model"); plt.xlabel("number of batches"); plt.ylabel("loss")
plt.legend(); plt.show()

# 5. Largest weight and bias in each layer
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
n_layers = len(models["sigmoid"].layers)
idx, bw = np.arange(n_layers), 0.35
for ax, (what, pos) in zip(axes, [("weight", 0), ("bias", 1)]):
    for k, (act, color) in enumerate([("sigmoid", "b"), ("relu", "r")]):
        vals = [np.max(l.get_weights()[pos]) for l in models[act].layers]
        ax.bar(idx + k * bw, vals, bw, label=act, color=color, alpha=0.4)
    ax.set_title(f"Maximum {what} per layer"); ax.set_xlabel("layer number"); ax.legend()
plt.tight_layout(); plt.show()


## Ex-6  Vehicle type recognition
**Aim:** Recognise vehicle types with a CNN built by transfer learning from VGG16.

In [ ]:
# Ex-6  Vehicle type recognition: transfer learning with VGG16
import os
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# ---- Set this to your dataset folder: one sub-folder per vehicle class ----
# Kaggle notebook : "/kaggle/input/vehicle-type-recognition/Dataset"
# Colab           : upload/unzip the dataset and point to it, e.g. "/content/Dataset"
DATASET_PATH = "/content/Dataset"
IMG_SIZE, BATCH, EPOCHS, SEED = (224, 224), 32, 15, 42

if not os.path.isdir(DATASET_PATH):
    raise FileNotFoundError(f"{DATASET_PATH} not found. Upload the vehicle dataset and fix DATASET_PATH.")
keras.utils.set_random_seed(SEED)

# 1. Load images, 80/20 train/validation split, one-hot labels
common = dict(validation_split=0.2, seed=SEED, image_size=IMG_SIZE, batch_size=BATCH, label_mode="categorical")
train_ds = keras.utils.image_dataset_from_directory(DATASET_PATH, subset="training", **common)
val_ds = keras.utils.image_dataset_from_directory(DATASET_PATH, subset="validation", **common)   # same seed => disjoint split
class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Classes:", class_names)
train_ds = train_ds.prefetch(2); val_ds = val_ds.prefetch(2)

# 2. Look at a few training images
plt.figure(figsize=(8, 8))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        plt.subplot(3, 3, i + 1); plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(np.argmax(labels[i]))]); plt.axis("off")
plt.show()

# 3. Model: augmentation -> frozen VGG16 base -> new classifier head
base = keras.applications.VGG16(weights="imagenet", include_top=False, input_shape=IMG_SIZE + (3,))
base.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomRotation(0.1)(x)
x = layers.RandomZoom(0.2)(x)
x = layers.RandomTranslation(0.1, 0.1)(x)
x = keras.applications.vgg16.preprocess_input(x)
x = base(x, training=False)
x = layers.Flatten()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inputs, outputs)

model.compile(loss="categorical_crossentropy",
              optimizer=keras.optimizers.RMSprop(learning_rate=1e-4), metrics=["accuracy"])
model.summary()

# 4. Train
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history["accuracy"], label="train"); ax[0].plot(history.history["val_accuracy"], label="val")
ax[0].set_title("Accuracy"); ax[0].legend()
ax[1].plot(history.history["loss"], label="train"); ax[1].plot(history.history["val_loss"], label="val")
ax[1].set_title("Loss"); ax[1].legend()
plt.show()

# 5. Evaluate: accuracy, per-class report and confusion matrix
loss, acc = model.evaluate(val_ds, verbose=0)
print(f"Validation accuracy: {acc:.4f}")
y_true, y_pred = [], []                      # collect labels and predictions in the SAME pass
for images, labels in val_ds:
    y_pred.extend(np.argmax(model.predict(images, verbose=0), axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1))
y_true, y_pred = np.array(y_true), np.array(y_pred)
labels = list(range(NUM_CLASSES))
print(classification_report(y_true, y_pred, labels=labels, target_names=class_names, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=labels), display_labels=class_names).plot(xticks_rotation=45)
plt.show()

# 6. Predictions on validation images
plt.figure(figsize=(10, 10))
for images, labels in val_ds.take(1):
    probs = model.predict(images, verbose=0)
    for i in range(min(9, len(images))):
        plt.subplot(3, 3, i + 1); plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(f"True: {class_names[int(np.argmax(labels[i]))]}\nPred: {class_names[int(np.argmax(probs[i]))]}")
        plt.axis("off")
plt.tight_layout(); plt.show()


## Ex-7  Diabetic Retinopathy
**Aim:** Classify retinal images into 5 severity levels with a CNN and data augmentation.

In [ ]:
# Ex-7  Diabetic Retinopathy detection (5-class CNN on retinal images)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ---- Set these paths (Kaggle "Diabetic Retinopathy Detection": images + trainLabels.csv) ----
IMG_DIR = "/content/dr_images"             # folder containing the .jpeg retina images
LABELS_CSV = "/content/trainLabels.csv"    # columns: image, level
WIDTH = HEIGHT = 128
NUM_CLASSES, EPOCHS, BS, INIT_LR = 5, 15, 32, 1e-3
CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
keras.utils.set_random_seed(10)

# 1. Read labels, keep only the images that are actually on disk (works with the 1000-image sample)
df = pd.read_csv(LABELS_CSV)
df["path"] = df["image"].apply(lambda n: os.path.join(IMG_DIR, n + ".jpeg"))
df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)
df["PatientID"] = df["image"].str.replace(r"_(left|right)$", "", regex=True)
print(len(df), "images from", df["PatientID"].nunique(), "patients")
print(df["level"].value_counts().sort_index().rename(dict(enumerate(CLASS_NAMES))))

# 2. Split BY PATIENT 75:25 so both eyes of a patient never sit in different sets
train_ids, val_ids = train_test_split(df["PatientID"].unique(), test_size=0.25, random_state=10)
train_df = df[df["PatientID"].isin(train_ids)].reset_index(drop=True)
val_df = df[df["PatientID"].isin(val_ids)].reset_index(drop=True)
print("train:", len(train_df), "| validation:", len(val_df))

# 3. tf.data pipeline: read -> resize -> scale to [0, 1] -> one-hot
def load(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (HEIGHT, WIDTH)) / 255.0
    return img, tf.one_hot(label, NUM_CLASSES)

def make_ds(frame, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((frame["path"].values, frame["level"].values.astype("int32")))
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).cache()
    if shuffle:
        ds = ds.shuffle(len(frame), seed=10)
    return ds.batch(BS).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds = make_ds(train_df, True), make_ds(val_df, False)

# 4. CNN with built-in augmentation (active only while training)
model = keras.Sequential([
    keras.Input(shape=(HEIGHT, WIDTH, 3)),
    layers.RandomRotation(30 / 360), layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.2), layers.RandomFlip("horizontal"),

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(2), layers.Dropout(0.25),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(2), layers.Dropout(0.25),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(2), layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(512, activation="relu"), layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation="softmax"),
])
steps = max(1, len(train_df) // BS)
lr = keras.optimizers.schedules.ExponentialDecay(INIT_LR, decay_steps=steps, decay_rate=0.95)
model.compile(loss="categorical_crossentropy", optimizer=keras.optimizers.Adam(lr), metrics=["accuracy"])
model.summary()

# 5. Train, save, plot
H = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
model.save("dr_model.keras")

plt.style.use("ggplot")
plt.figure()
for k in ["loss", "val_loss", "accuracy", "val_accuracy"]:
    plt.plot(H.history[k], label=k)
plt.title("Training loss and accuracy: diabetic retinopathy detection")
plt.xlabel("Epoch #"); plt.ylabel("Loss / Accuracy"); plt.legend(loc="lower left")
plt.savefig("plot.png"); plt.show()

# 6. Evaluate
y_true = np.concatenate([np.argmax(l.numpy(), axis=1) for _, l in val_ds])
y_pred = np.argmax(model.predict(val_ds, verbose=0), axis=1)
print(classification_report(y_true, y_pred, labels=range(NUM_CLASSES),
                            target_names=CLASS_NAMES, zero_division=0))


## Ex-8  Different optimizers
**Aim:** Train the same network with several optimizers, compare the results and pick the best.

In [ ]:
# Ex-8  Compare different optimizers (Wine Quality regression)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 2022
keras.utils.set_random_seed(SEED)

def load_wine():
    urls = ["https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv",
            "https://raw.githubusercontent.com/shrikant-temburwar/Wine-Quality-Dataset/master/winequality-red.csv"]
    for u in urls:
        try:
            return pd.read_csv(u, sep=";")
        except Exception as e:
            print("could not read", u, "->", type(e).__name__)
    raise RuntimeError("Wine data not reachable")

data = load_wine()
y = data["quality"].values.astype("float32")
X = data.drop(["quality"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=SEED)

scaler = StandardScaler().fit(X_train)            # scale: unscaled wine data makes SGD diverge
X_train, X_val, X_test = [scaler.transform(a).astype("float32") for a in (X_train, X_val, X_test)]
print("NaN/inf in data:", any(not np.isfinite(a).all() for a in (X_train, X_val, X_test, y_train, y_val, y_test)))

def create_model():
    return keras.Sequential([
        keras.Input(shape=(X_train.shape[1],)),
        layers.Dense(100, activation="relu"), layers.Dense(50, activation="relu"),
        layers.Dense(25, activation="relu"), layers.Dense(10, activation="relu"),
        layers.Dense(1, activation="linear"),
    ])

# A fresh optimizer object for every run
opts = {
    "sgd":          lambda: keras.optimizers.SGD(),
    "sgd-0001":     lambda: keras.optimizers.SGD(learning_rate=1e-4),
    "adam":         lambda: keras.optimizers.Adam(),
    "adadelta":     lambda: keras.optimizers.Adadelta(),
    "rmsprop":      lambda: keras.optimizers.RMSprop(),
    "rmsprop-0001": lambda: keras.optimizers.RMSprop(learning_rate=1e-4),
    "nadam":        lambda: keras.optimizers.Nadam(),
    "adamax":       lambda: keras.optimizers.Adamax(),
}

BATCH, MAX_EPOCHS = 128, 300
results, curves = [], {}
for name, make_opt in opts.items():
    keras.utils.set_random_seed(SEED)
    model = create_model()
    model.compile(loss="mse", optimizer=make_opt())
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=30, restore_best_weights=True)
    hist = model.fit(X_train, y_train, batch_size=BATCH, epochs=MAX_EPOCHS,
                     validation_data=(X_val, y_val), callbacks=[stop], verbose=0)
    best_epoch = int(np.argmin(hist.history["val_loss"]))
    test_mse = float(model.evaluate(X_test, y_test, verbose=0))
    test_acc = float(np.mean(np.round(model.predict(X_test, verbose=0).ravel()) == y_test))
    results.append([name, best_epoch + 1, hist.history["val_loss"][best_epoch], test_mse, test_acc])
    curves[name] = hist.history["val_loss"]
    print(f"{name:<13} best epoch {best_epoch + 1:>3} | val MSE {results[-1][2]:.4f} | test MSE {test_mse:.4f}")

res = pd.DataFrame(results, columns=["optimizer", "best_epoch", "val_mse", "test_mse", "test_rounded_acc"])
print("\n", res.sort_values("test_mse").to_string(index=False))
print("\nBest optimizer (lowest test MSE):", res.loc[res["test_mse"].idxmin(), "optimizer"])

plt.figure(figsize=(9, 5))
for name, v in curves.items():
    plt.plot(v, label=name)
plt.ylim(0, 2); plt.xlabel("epochs"); plt.ylabel("validation MSE")
plt.title("Optimizer comparison"); plt.legend(); plt.show()


## Ex-9  Improving generalisation with regularisation
**Aim:** Reduce overfitting on the Bike Sharing data with L2 weight regularisation.

In [ ]:
# Ex-9  Improving generalisation with L2 regularisation (Bike Sharing, hourly)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers

keras.utils.set_random_seed(2022)

import os, io, zipfile, urllib.request

def load_bike():
    if os.path.exists("hour.csv"):
        return pd.read_csv("hour.csv")
    url = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
    try:
        raw = urllib.request.urlopen(url, timeout=60).read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            return pd.read_csv(z.open("hour.csv"))
    except Exception as e:
        print("download failed:", type(e).__name__, "- upload hour.csv from the UCI Bike Sharing dataset")
        from google.colab import files        # Colab only
        files.upload()
        return pd.read_csv("hour.csv")

def prepare_bike(data):
    ohe_features = ["season", "weathersit", "mnth", "hr", "weekday"]
    for f in ohe_features:
        data = pd.concat([data, pd.get_dummies(data[f], prefix=f, drop_first=False, dtype=float)], axis=1)
    drop = ["instant", "dteday", "season", "weathersit", "weekday", "atemp", "mnth",
            "workingday", "hr", "casual", "registered"]
    data = data.drop(drop, axis=1)
    for f in ["cnt", "temp", "hum", "windspeed"]:                 # standardise
        data[f] = (data[f] - data[f].mean()) / data[f].std()
    test_data, data = data[-31 * 24:], data[:-31 * 24]            # last month = test (744 rows)
    feats, targ = data.drop(["cnt"], axis=1), data["cnt"]
    X_train, y_train = feats[:-30 * 24], targ[:-30 * 24]          # last 30 days of the rest = validation
    X_val, y_val = feats[-30 * 24:], targ[-30 * 24:]
    X_test, y_test = test_data.drop(["cnt"], axis=1), test_data["cnt"]
    f32 = lambda a: a.values.astype("float32")
    return f32(X_train), f32(y_train), f32(X_val), f32(y_val), f32(X_test), f32(y_test)

X_train, y_train, X_val, y_val, X_test, y_test = prepare_bike(load_bike())
print("train", X_train.shape, "| validation", X_val.shape, "| test", X_test.shape)

N_EPOCHS, BATCH = 2000, 1024          # the manual uses 4000 epochs; 2000 shows the same behaviour faster

def build(l2=None):
    reg = keras.regularizers.l2(l2) if l2 else None
    m = keras.Sequential([
        keras.Input(shape=(X_train.shape[1],)),
        layers.Dense(250, activation="relu", kernel_regularizer=reg),
        layers.Dense(150, activation="relu"),
        layers.Dense(50, activation="relu"),
        layers.Dense(25, activation="relu", kernel_regularizer=reg),
        layers.Dense(1, activation="linear"),
    ])
    m.compile(loss="mse", optimizer="sgd", metrics=["mse"])
    return m

runs = {}
for name, l2 in [("no regularisation", None), ("L2 = 0.005", 0.005)]:
    keras.utils.set_random_seed(2022)
    h = build(l2).fit(X_train, y_train, validation_data=(X_val, y_val),
                      batch_size=BATCH, epochs=N_EPOCHS, verbose=0)
    runs[name] = h.history
    print(f"{name:<18} min validation loss: {min(h.history['val_loss']):.4f} "
          f"after {int(np.argmin(h.history['val_loss'])) + 1} epochs | "
          f"final validation loss: {h.history['val_loss'][-1]:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for a, (name, h) in zip(ax, runs.items()):
    a.plot(h["loss"], label="training"); a.plot(h["val_loss"], label="validation")
    a.set_title(name); a.set_xlabel("epochs"); a.legend()
ax[0].set_ylabel("loss"); plt.tight_layout(); plt.show()


## Ex-10  Dropout to prevent overfitting
**Aim:** Reduce overfitting on the Bike Sharing data by adding dropout layers.

In [ ]:
# Ex-10  Adding dropout to prevent overfitting (Bike Sharing, hourly)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers

keras.utils.set_random_seed(2022)

import os, io, zipfile, urllib.request

def load_bike():
    if os.path.exists("hour.csv"):
        return pd.read_csv("hour.csv")
    url = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
    try:
        raw = urllib.request.urlopen(url, timeout=60).read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            return pd.read_csv(z.open("hour.csv"))
    except Exception as e:
        print("download failed:", type(e).__name__, "- upload hour.csv from the UCI Bike Sharing dataset")
        from google.colab import files        # Colab only
        files.upload()
        return pd.read_csv("hour.csv")

def prepare_bike(data):
    ohe_features = ["season", "weathersit", "mnth", "hr", "weekday"]
    for f in ohe_features:
        data = pd.concat([data, pd.get_dummies(data[f], prefix=f, drop_first=False, dtype=float)], axis=1)
    drop = ["instant", "dteday", "season", "weathersit", "weekday", "atemp", "mnth",
            "workingday", "hr", "casual", "registered"]
    data = data.drop(drop, axis=1)
    for f in ["cnt", "temp", "hum", "windspeed"]:                 # standardise
        data[f] = (data[f] - data[f].mean()) / data[f].std()
    test_data, data = data[-31 * 24:], data[:-31 * 24]            # last month = test (744 rows)
    feats, targ = data.drop(["cnt"], axis=1), data["cnt"]
    X_train, y_train = feats[:-30 * 24], targ[:-30 * 24]          # last 30 days of the rest = validation
    X_val, y_val = feats[-30 * 24:], targ[-30 * 24:]
    X_test, y_test = test_data.drop(["cnt"], axis=1), test_data["cnt"]
    f32 = lambda a: a.values.astype("float32")
    return f32(X_train), f32(y_train), f32(X_val), f32(y_val), f32(X_test), f32(y_test)

X_train, y_train, X_val, y_val, X_test, y_test = prepare_bike(load_bike())
print("train", X_train.shape, "| validation", X_val.shape, "| test", X_test.shape)

N_EPOCHS, BATCH, RATE = 2000, 1024, 0.20

def build(dropout=0.0):
    m = keras.Sequential([keras.Input(shape=(X_train.shape[1],))])
    for units in [250, 150, 50, 25]:
        m.add(layers.Dense(units, activation="relu"))
        if dropout:
            m.add(layers.Dropout(dropout))      # randomly zeroes units during training only
    m.add(layers.Dense(1, activation="linear"))
    m.compile(loss="mse", optimizer="sgd", metrics=["mse"])
    return m

runs = {}
for name, rate in [("no dropout", 0.0), (f"dropout {RATE}", RATE)]:
    keras.utils.set_random_seed(2022)
    model = build(rate)
    h = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                  batch_size=BATCH, epochs=N_EPOCHS, verbose=0)
    runs[name] = h.history
    test_mse = model.evaluate(X_test, y_test, verbose=0)[0]
    print(f"{name:<14} min validation loss: {min(h.history['val_loss']):.4f} "
          f"after {int(np.argmin(h.history['val_loss'])) + 1} epochs | test MSE: {test_mse:.4f}")

model.summary()

fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for a, (name, h) in zip(ax, runs.items()):
    a.plot(h["loss"], label="training"); a.plot(h["val_loss"], label="validation")
    a.set_title(name); a.set_xlabel("epochs"); a.legend()
ax[0].set_ylabel("loss"); plt.tight_layout(); plt.show()


## Ex-11  Image augmentation
**Aim:** Apply resizing, flips, rotation, brightness, contrast, crop and custom augmentations to images.

In [ ]:
# Ex-11  Image augmentation with Keras preprocessing layers and tf.image
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds      # Colab: already installed
import keras
from keras import layers

# 1. Load the TF Flowers dataset and pick an image
(train_ds, val_ds, test_ds), metadata = tfds.load(
    "tf_flowers", split=["train[:80%]", "train[80%:90%]", "train[90%:]"],
    with_info=True, as_supervised=True)
get_label_name = metadata.features["label"].int2str

fig = plt.figure(figsize=(14, 3))
for i, (img, lab) in enumerate(train_ds.take(6)):
    fig.add_subplot(1, 6, i + 1); plt.imshow(img); plt.axis("off"); plt.title(get_label_name(lab))
plt.show()

image, label = next(iter(train_ds))
print("Image shape:", image.shape, "| label:", get_label_name(label))

def visualize(original, augmented, title="Augmented image"):
    plt.figure(figsize=(7, 3))
    plt.subplot(1, 2, 1); plt.title("Original image"); plt.imshow(original); plt.axis("off")
    plt.subplot(1, 2, 2); plt.title(title); plt.imshow(augmented); plt.axis("off")
    plt.show()

# 2. Resize and rescale
IMG_SIZE = 180
resize_and_rescale = keras.Sequential([layers.Resizing(IMG_SIZE, IMG_SIZE), layers.Rescaling(1. / 255)])
result = resize_and_rescale(tf.cast(image, tf.float32))
plt.imshow(result); plt.axis("off"); plt.title("Resized + rescaled"); plt.show()

# 3. Random flip + rotation (random layers only act when training=True)
data_augmentation = keras.Sequential([layers.RandomFlip("horizontal_and_vertical"), layers.RandomRotation(0.4)])
plt.figure(figsize=(8, 5))
for i in range(6):
    aug = data_augmentation(tf.expand_dims(tf.cast(image, tf.float32) / 255., 0), training=True)
    plt.subplot(2, 3, i + 1); plt.imshow(aug[0]); plt.axis("off")
plt.suptitle("RandomFlip + RandomRotation"); plt.show()

# 4. Custom augmentation: random colour inversion
def random_invert_img(x, p=0.5):
    return tf.cond(tf.random.uniform([]) < p, lambda: 255 - x, lambda: x)

img_f = tf.cast(image, tf.float32)
plt.figure(figsize=(8, 8))
for i in range(9):
    plt.subplot(3, 3, i + 1); plt.imshow(tf.cast(random_invert_img(img_f), tf.uint8)); plt.axis("off")
plt.suptitle("Random invert"); plt.show()

# 5. Deterministic and stateless-random tf.image operations
visualize(image, tf.image.flip_left_right(image), "Flipped")
gray = tf.image.rgb_to_grayscale(image)
visualize(image, tf.squeeze(gray), "Grayscale"); 
visualize(image, tf.image.adjust_saturation(image, 4), "Saturation x4")
visualize(image, tf.image.adjust_brightness(image, 0.4), "Brightness +0.4")
visualize(image, tf.image.central_crop(image, central_fraction=0.5), "Central crop 50%")
visualize(image, tf.image.rot90(image), "Rotated 90")

h, w = int(image.shape[0] * 0.7), int(image.shape[1] * 0.7)
for i in range(3):
    seed = (i, 0)
    visualize(image, tf.image.stateless_random_brightness(image, max_delta=0.95, seed=seed), f"Random brightness (seed {i})")
    visualize(image, tf.image.stateless_random_contrast(image, lower=0.1, upper=0.9, seed=seed), f"Random contrast (seed {i})")
    visualize(image, tf.image.stateless_random_crop(image, size=[h, w, 3], seed=seed), f"Random crop (seed {i})")

# 6. Use augmentation in a real input pipeline (training set only)
def prepare(ds, augment=False, batch=32):
    ds = ds.map(lambda x, y: (resize_and_rescale(tf.cast(x, tf.float32)), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch)
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_aug = prepare(train_ds, augment=True)
xb, yb = next(iter(train_aug))
print("Augmented training batch:", xb.shape, "pixel range", float(tf.reduce_min(xb)), "to", float(tf.reduce_max(xb)))


## Ex-12(a)  LeNet-5
**Aim:** Implement the LeNet-5 CNN (trained on MNIST as in the manual).

In [ ]:
# Ex-12(a)  LeNet-5 (the manual runs it on MNIST)
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

keras.utils.set_random_seed(42)

# 1. Data: normalise and add the channel axis -> (28, 28, 1)
(train_x, train_y), (test_x, test_y) = keras.datasets.mnist.load_data()
train_x = np.expand_dims(train_x.astype("float32") / 255.0, -1)
test_x = np.expand_dims(test_x.astype("float32") / 255.0, -1)
val_x, val_y = train_x[:5000], train_y[:5000]

# 2. LeNet-5: C1 - S2 - C3 - S4 - C5 - F6 - output
lenet = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(6, kernel_size=5, strides=1, activation="tanh", padding="same"),   # C1
    layers.AveragePooling2D(pool_size=2),                                                      # S2
    layers.Conv2D(16, kernel_size=5, strides=1, activation="tanh", padding="valid"),  # C3
    layers.AveragePooling2D(pool_size=2),                                                      # S4
    layers.Conv2D(120, kernel_size=5, strides=1, activation="tanh", padding="valid"), # C5
    layers.Flatten(),
    layers.Dense(84, activation="tanh"),                                              # F6
    layers.Dense(10, activation="softmax"),                                           # output
])
lenet.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
lenet.summary()

# 3. Train and evaluate
history = lenet.fit(train_x, train_y, epochs=5, validation_data=(val_x, val_y))
test_loss, test_acc = lenet.evaluate(test_x, test_y, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")

# 4. Curves and a few predictions
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history.history["accuracy"], label="train"); ax[0].plot(history.history["val_accuracy"], label="val")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("epochs"); ax[0].legend()
ax[1].plot(history.history["loss"], label="train"); ax[1].plot(history.history["val_loss"], label="val")
ax[1].set_title("Loss"); ax[1].set_xlabel("epochs"); ax[1].legend()
plt.show()

pred = np.argmax(lenet.predict(test_x[:10], verbose=0), axis=1)
plt.figure(figsize=(12, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1); plt.imshow(test_x[i, :, :, 0], cmap="gray"); plt.axis("off")
    plt.title(f"{pred[i]} ({test_y[i]})", fontsize=8)
plt.suptitle("Predicted (true)"); plt.show()


## Ex-12(b)  AlexNet
**Aim:** Implement the AlexNet CNN (trained on CIFAR-10 resized to 227x227 as in the manual).

In [ ]:
# Ex-12(b)  AlexNet (the manual trains it on CIFAR-10 resized to 227x227). Use a GPU runtime.
import os, time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

keras.utils.set_random_seed(42)
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

# 1. Data: hold out 5000 training images for validation
(train_images, train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()
val_images, val_labels = train_images[:5000], train_labels[:5000]
train_images, train_labels = train_images[5000:], train_labels[5000:]

plt.figure(figsize=(12, 3))
for i in range(5):
    plt.subplot(1, 5, i + 1); plt.imshow(train_images[i]); plt.axis("off"); plt.title(CLASS_NAMES[int(train_labels[i][0])])
plt.show()

def process_images(image, label):
    image = tf.image.per_image_standardization(tf.cast(image, tf.float32))   # mean 0, std 1
    image = tf.image.resize(image, (227, 227))                                # AlexNet input size
    return image, tf.squeeze(label)

def make_ds(x, y, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(len(x), seed=42)
    return ds.map(process_images, num_parallel_calls=tf.data.AUTOTUNE).batch(32, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_images, train_labels, True)
validation_ds = make_ds(val_images, val_labels, False)
test_ds = make_ds(test_images, test_labels, False)
print("train / val / test images:", len(train_images), len(val_images), len(test_images))

# 2. AlexNet: 5 conv layers + 3 dense layers, with BatchNorm and Dropout
model = keras.Sequential([
    keras.Input(shape=(227, 227, 3)),
    layers.Conv2D(96, (11, 11), strides=(4, 4), activation="relu"),
    layers.BatchNormalization(), layers.MaxPool2D((3, 3), strides=(2, 2)),
    layers.Conv2D(256, (5, 5), padding="same", activation="relu"),
    layers.BatchNormalization(), layers.MaxPool2D((3, 3), strides=(2, 2)),
    layers.Conv2D(384, (3, 3), padding="same", activation="relu"), layers.BatchNormalization(),
    layers.Conv2D(384, (3, 3), padding="same", activation="relu"), layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), padding="same", activation="relu"), layers.BatchNormalization(),
    layers.MaxPool2D((3, 3), strides=(2, 2)),
    layers.Flatten(),
    layers.Dense(4096, activation="relu"), layers.Dropout(0.5),
    layers.Dense(4096, activation="relu"), layers.Dropout(0.5),
    layers.Dense(10, activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=keras.optimizers.SGD(learning_rate=0.001, momentum=0.9), metrics=["accuracy"])
model.summary()

# 3. Train with TensorBoard logging (view with:  %load_ext tensorboard ; %tensorboard --logdir logs)
run_logdir = os.path.join("logs", time.strftime("run_%Y_%m_%d-%H_%M_%S"))
history = model.fit(train_ds, epochs=5, validation_data=validation_ds,
                    callbacks=[keras.callbacks.TensorBoard(run_logdir)])

# 4. Evaluate and visualise
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history.history["accuracy"], label="train"); ax[0].plot(history.history["val_accuracy"], label="val")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("epochs"); ax[0].legend()
ax[1].plot(history.history["loss"], label="train"); ax[1].plot(history.history["val_loss"], label="val")
ax[1].set_title("Loss"); ax[1].set_xlabel("epochs"); ax[1].legend()
plt.show()

xb, yb = next(iter(test_ds))
pb = np.argmax(model.predict(xb, verbose=0), axis=1)
plt.figure(figsize=(12, 3))
for i in range(8):
    img = xb[i].numpy(); img = (img - img.min()) / (img.max() - img.min())
    plt.subplot(1, 8, i + 1); plt.imshow(img); plt.axis("off")
    plt.title(f"{CLASS_NAMES[pb[i]]}\n({CLASS_NAMES[int(yb[i])]})", fontsize=7)
plt.suptitle("Predicted (true)"); plt.show()


## Ex-13(a)  RNN
**Aim:** Implement a recurrent neural network that classifies MNIST digits row by row.

In [ ]:
# Ex-13(a)  Recurrent neural network: classify MNIST digits by reading each image row by row
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

keras.utils.set_random_seed(42)

# 1. Data: each 28x28 image is a sequence of 28 time steps with 28 features
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
print(x_train.shape, x_train[0].shape)

# 2. Stacked SimpleRNN (tanh) -> dense classifier
model = keras.Sequential([
    keras.Input(shape=x_train.shape[1:]),
    layers.SimpleRNN(128, return_sequences=True),
    layers.Dropout(0.2),
    layers.SimpleRNN(128),
    layers.Dropout(0.1),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(10, activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),   # clipnorm tames exploding gradients
              metrics=["accuracy"])
model.summary()

# 3. Train and evaluate
history = model.fit(x_train, y_train, epochs=5, batch_size=128, validation_data=(x_test, y_test))
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {loss:.4f} | Test accuracy: {acc:.4f}")

# 4. Curves and predictions
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history.history["accuracy"], label="train"); ax[0].plot(history.history["val_accuracy"], label="val")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("epochs"); ax[0].legend()
ax[1].plot(history.history["loss"], label="train"); ax[1].plot(history.history["val_loss"], label="val")
ax[1].set_title("Loss"); ax[1].set_xlabel("epochs"); ax[1].legend()
plt.show()

pred = np.argmax(model.predict(x_test[:10], verbose=0), axis=1)
plt.figure(figsize=(12, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1); plt.imshow(x_test[i], cmap="gray"); plt.axis("off")
    plt.title(f"{pred[i]} ({y_test[i]})", fontsize=8)
plt.suptitle("Predicted (true)"); plt.show()


## Ex-13(b)  LSTM
**Aim:** Implement an LSTM to forecast the AirPassengers time series.

In [ ]:
# Ex-13(b)  LSTM time-series forecasting: AirPassengers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

keras.utils.set_random_seed(7)

# 1. Load the monthly passenger counts
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
dataframe = pd.read_csv(URL, usecols=[1], engine="python")        # AirPassengers.csv works too
dataset = dataframe.values.astype("float32")
plt.plot(dataset); plt.title("International airline passengers"); plt.xlabel("month"); plt.show()

# 2. Normalise to [0, 1] and split 67% / 33% in time order
scaler = MinMaxScaler(feature_range=(0, 1))
dataset = scaler.fit_transform(dataset)
train_size = int(len(dataset) * 0.67)
train, test = dataset[:train_size], dataset[train_size:]
print("train / test points:", len(train), len(test))

# 3. Turn the series into (X = previous `look_back` values, y = next value)
def create_dataset(data, look_back=1):
    X, Y = [], []
    for i in range(len(data) - look_back - 1):
        X.append(data[i:i + look_back, 0]); Y.append(data[i + look_back, 0])
    return np.array(X), np.array(Y)

look_back = 1
trainX, trainY = create_dataset(train, look_back)
testX, testY = create_dataset(test, look_back)
trainX = trainX.reshape(trainX.shape[0], 1, trainX.shape[1])      # [samples, time steps, features]
testX = testX.reshape(testX.shape[0], 1, testX.shape[1])

# 4. LSTM(4) -> Dense(1)
model = keras.Sequential([
    keras.Input(shape=(1, look_back)),
    layers.LSTM(4),
    layers.Dense(1),
])
model.compile(loss="mean_squared_error", optimizer="adam")
model.summary()
history = model.fit(trainX, trainY, epochs=100, batch_size=1, verbose=0)
print("final training loss:", round(history.history["loss"][-1], 6))

# 5. Predict, undo the scaling, report RMSE
trainPredict = scaler.inverse_transform(model.predict(trainX, verbose=0))
testPredict = scaler.inverse_transform(model.predict(testX, verbose=0))
trainY_inv = scaler.inverse_transform([trainY]); testY_inv = scaler.inverse_transform([testY])
print("Train Score: %.2f RMSE" % np.sqrt(mean_squared_error(trainY_inv[0], trainPredict[:, 0])))
print("Test Score: %.2f RMSE" % np.sqrt(mean_squared_error(testY_inv[0], testPredict[:, 0])))

# 6. Plot actual vs predicted over the whole series
trainPlot = np.full_like(dataset, np.nan); trainPlot[look_back:len(trainPredict) + look_back] = trainPredict
testPlot = np.full_like(dataset, np.nan)
testPlot[len(trainPredict) + (look_back * 2) + 1:len(dataset) - 1] = testPredict
plt.figure(figsize=(10, 4))
plt.plot(scaler.inverse_transform(dataset), label="actual")
plt.plot(trainPlot, label="train prediction"); plt.plot(testPlot, label="test prediction")
plt.title("LSTM forecast: AirPassengers"); plt.xlabel("month"); plt.legend(); plt.show()


## Ex-14  GRU
**Aim:** Implement a GRU (compared with an LSTM) to forecast IBM stock prices.

In [ ]:
# Ex-14  GRU (compared with LSTM) for stock-price forecasting: IBM "High" price
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

keras.utils.set_random_seed(42)

# 1. Data: IBM daily prices 2006-2018 (cached to IBM.csv). Train: up to 2016, test: 2017.
def load_ibm():
    if os.path.exists("IBM.csv"):
        return pd.read_csv("IBM.csv", index_col="Date", parse_dates=["Date"])
    try:
        import yfinance as yf                                      # Colab:  !pip -q install yfinance
        df = yf.download("IBM", start="2006-01-01", end="2018-01-01", auto_adjust=False, progress=False)
        df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
        df = df[["Open", "High", "Low", "Close", "Volume"]]
        df.to_csv("IBM.csv")
        return df
    except Exception as e:
        print("download failed:", type(e).__name__, "- upload IBM.csv (Date, Open, High, Low, Close, Volume)")
        from google.colab import files
        files.upload()
        return pd.read_csv("IBM.csv", index_col="Date", parse_dates=["Date"])

dataset = load_ibm()
training_set = dataset[:"2016"]["High"].values.reshape(-1, 1)
test_set = dataset["2017":]["High"].values.reshape(-1, 1)

dataset["High"][:"2016"].plot(figsize=(14, 4), legend=True)
dataset["High"]["2017":].plot(legend=True)
plt.legend(["Training set (up to 2016)", "Test set (2017)"]); plt.title("IBM stock price (High)"); plt.show()

# 2. Scale and build windows: 60 previous days -> next day
sc = MinMaxScaler(feature_range=(0, 1))
training_scaled = sc.fit_transform(training_set)
WINDOW = 60
X_train = np.array([training_scaled[i - WINDOW:i, 0] for i in range(WINDOW, len(training_scaled))])
y_train = np.array([training_scaled[i, 0] for i in range(WINDOW, len(training_scaled))])
X_train = X_train.reshape(X_train.shape[0], WINDOW, 1)

total = pd.concat((dataset["High"][:"2016"], dataset["High"]["2017":]), axis=0)
inputs = sc.transform(total[len(total) - len(test_set) - WINDOW:].values.reshape(-1, 1))
X_test = np.array([inputs[i - WINDOW:i, 0] for i in range(WINDOW, WINDOW + len(test_set))])
X_test = X_test.reshape(X_test.shape[0], WINDOW, 1)

def stacked(cell):
    m = keras.Sequential([keras.Input(shape=(WINDOW, 1))])
    for i in range(4):
        m.add(cell(50, return_sequences=(i < 3)))
        m.add(layers.Dropout(0.2))
    m.add(layers.Dense(1))
    return m

def rmse(real, pred):
    return math.sqrt(mean_squared_error(real, pred))

# 3. Same stacked architecture and same optimizer (RMSprop) for LSTM and GRU, so the comparison is fair
#    (the manual trained the GRU with SGD + momentum, which needs many more epochs to converge)
EPOCHS = 20   # the manual uses 5; more epochs lets both models converge
lstm_model = stacked(layers.LSTM)
lstm_model.compile(optimizer="rmsprop", loss="mean_squared_error")
lstm_model.fit(X_train, y_train, epochs=EPOCHS, batch_size=32, verbose=2)
lstm_pred = sc.inverse_transform(lstm_model.predict(X_test, verbose=0))

gru_model = stacked(layers.GRU)
gru_model.compile(optimizer="rmsprop", loss="mean_squared_error")
gru_model.summary()
gru_model.fit(X_train, y_train, epochs=EPOCHS, batch_size=32, verbose=2)
gru_pred = sc.inverse_transform(gru_model.predict(X_test, verbose=0))

# 4. Compare
print("LSTM RMSE:", round(rmse(test_set, lstm_pred), 4))
print("GRU  RMSE:", round(rmse(test_set, gru_pred), 4))

plt.figure(figsize=(12, 4))
plt.plot(test_set, color="red", label="Real IBM stock price")
plt.plot(lstm_pred, color="blue", label="LSTM prediction")
plt.plot(gru_pred, color="green", label="GRU prediction")
plt.title("IBM stock price prediction"); plt.xlabel("Time"); plt.ylabel("IBM stock price"); plt.legend(); plt.show()
